# Solar Filament Segmentation Challenge 2026 — Submission Reproduction

**これは提出物の公式再現 notebook です。** 「すべて実行」で前処理 → 検出器学習 →
refiner 学習 → test 推論 → `submission.csv` 生成まで一通り通ります。
既存の `artifacts/grouped_folds.csv` と外部で凍結した SHA256 を
`CFG.GROUPED_FOLDS_SHA256` に設定してください。未設定・欠落・不整合なら
前処理前に停止し、新しい分割は生成しません。

## 実行環境

| 環境 | 手順 |
|---|---|
| Kaggle | 本 notebook に競技データ(`filament-segmentation-2026`)と重み dataset をアタッチして Run。GPU T4x2 以上推奨 |
| Colab | Google Drive に `solar_filament_v30` ルート(競技データ + 成果物)を配置して Run。A100/L4 推奨 |

## 重みの解決順序

1. Kaggle にアタッチされた重み dataset(`*yolo*fold{N}*best*.pt`, `*refiner*all*data*.pt`)
2. Drive artifacts(`solar_filament_v30/runs`, `artifacts/`)内の既存 checkpoint
3. どちらも無ければ **フル再学習**(YOLO 3-fold + refiner 3-fold + all-data fine-tune、A100 で約6-8時間)

## 確定構成(V50/V52 提出と同一)

- YOLO11m-seg @1280, conf=0.26, iou=0.55, retina_masks
- EdgeAttNet風 refiner(crop 320px, context 1.65, threshold 0.45)
- 包含率0.90でのdrop → 面積昇順 ownership(排他化)→ min_area 20
- fold2 単一モデル(OOF PQ最良、アンサンブルより計算1/3で同等以上)

## 選択した2つの提出の再現

最終選択は `prod`(上記構成)と `plain_mhsa_f2`(refiner を ablation の
plain-MHSA fold2 重みに差し替えた構成、アーキテクチャ差による private ヘッジ)。
`prod` は本 notebook をそのまま実行すれば再現できる。
`plain_mhsa_f2` は setup セルの `CFG.SUBMISSION_VARIANT` を `"plain_mhsa_f2"`
に変更して再実行する(重み dataset の `ablation_plain_mhsa_seed8407_fold2.pt`
を使用、出力は `submission_plain_mhsa_f2.csv`)。


## (A) セットアップ・データ読み込み・fold分割

In [ ]:
# [1/6] 環境検出・依存関係・設定
%pip install -q "ultralytics>=8.3,<9" pycocotools

import json, math, os, random, shutil, time
from collections import defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pycocotools import mask as mask_utils
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from ultralytics import YOLO

IS_KAGGLE = Path("/kaggle/input").exists()
if IS_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
    DATA_SEARCH_ROOTS = [Path("/kaggle/input")]
else:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_ROOT = Path("/content")

class CFG:
    SEED = 3407
    N_FOLDS = 3
    GROUPED_FOLDS_SHA256 = ""  # 外部で凍結した grouped_folds.csv の SHA256 が必須
    if IS_KAGGLE:
        DRIVE_ROOT = WORK_ROOT / "solar_filament_v30"  # 成果物の書き込み先(working)
        PREPROC_CACHE = None  # Kaggle には前処理キャッシュ無し → inline計算
    else:
        DRIVE_ROOT = Path("/content/drive/MyDrive/solar_filament_v30")
        PREPROC_CACHE = DRIVE_ROOT / "preprocessed_v30_q96"
    DATA_CACHE = DRIVE_ROOT / "competition_data"
    ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
    RUNS_ROOT = DRIVE_ROOT / "runs"
    YOLO_ROOT = DRIVE_ROOT / "yolo_datasets"
    LOCAL_ROOT = WORK_ROOT / "solar_master"
    PREPROC_DIR = LOCAL_ROOT / "preprocessed"
    # --- YOLO検出器(build_v30_prize_colab.pyと同一) ---
    YOLO_MODEL = "yolo11m-seg.pt"
    YOLO_IMGSZ = 1280
    YOLO_BATCH = 4
    YOLO_EPOCHS = 70
    YOLO_PATIENCE = 14
    YOLO_MASK_RATIO = 2
    # --- refiner(build_v35_refiner_folds12.py / build_v51_edgeattnet_refiner.pyと同一) ---
    CROP_SIZE = 320
    CROP_CONTEXT = 1.65
    REFINER_EPOCHS = 14
    REFINER_ALLDATA_EPOCHS = 5
    REFINER_LR = 2e-4
    REFINER_BATCH_CANDIDATES = (8, 6, 4, 2)
    # --- 確定後処理(README.md「現在地」節、V52提出構成) ---
    # "prod"(既定: EdgeAtt all-data refiner) / "plain_mhsa_f2"(第2提出:
    # ablation plain-MHSA fold2 refiner、重み dataset の
    # ablation_plain_mhsa_seed8407_fold2.pt が必要)
    SUBMISSION_VARIANT = "prod"
    SUBMISSION_CONF = 0.26
    REFINER_THRESHOLD = 0.45
    DROP_THRESHOLD = 0.90
    MIN_AREA = 20
    PREDICT_IOU = 0.55
    SINGLE_FOLD = 2  # V50/V52で確認済み

for path in (CFG.LOCAL_ROOT, CFG.PREPROC_DIR, CFG.ARTIFACT_ROOT, CFG.YOLO_ROOT, CFG.RUNS_ROOT):
    path.mkdir(parents=True, exist_ok=True)
random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)


def find_dataset_root():
    # Kaggle: /kaggle/input 以下、Colab: Drive competition_data 以下を探す
    for base in DATA_SEARCH_ROOTS if IS_KAGGLE else [CFG.DATA_CACHE]:
        matches = list(base.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
        if matches:
            return matches[0].parents[2]
    raise FileNotFoundError("競技データ(MAGFiLO annotations JSON)が見つかりません。"
                            "Kaggleでは競技データをアタッチ、ColabではDriveに配置してください")


def resolve_pretrained(patterns):
    """添付データセット/Drive成果物から事前学習済み重みを探す。見つかれば Path を返す。"""
    for base in DATA_SEARCH_ROOTS if IS_KAGGLE else [CFG.RUNS_ROOT, CFG.ARTIFACT_ROOT]:
        for pattern in patterns:
            hits = sorted(base.rglob(pattern))
            hits = [h for h in hits if h.is_file() and h.stat().st_size > 1_000_000]
            if hits:
                return hits[0]
    return None


assert torch.cuda.is_available(), "GPUランタイムを選択してください"
print({"env": "kaggle" if IS_KAGGLE else "colab",
       "GPU": torch.cuda.get_device_name(0)})


In [ ]:
# [2/6] データ読み込み・前処理・fold分割
def find_train_dir(data_root):
    matches = list(data_root.rglob("train_images"))
    if matches:
        return matches[0]
    # レイアウト差への耐性: annotation JSON と同階層から jpeg を探す
    return next(data_root.rglob("*train*"))


def robust_uint8(array, mask=None, low=0.5, high=99.5):
    values = array[mask] if mask is not None and mask.any() else array.reshape(-1)
    lo, hi = np.percentile(values, [low, high])
    return np.clip((array - lo) * 255.0 / max(hi - lo, 1e-6), 0, 255).astype(np.uint8)


def solar_three_channel(gray):
    # EdgeAttNet着想: 径方向/背景勾配を抑えて暗いフィラメントを浮かせつつ元観測も残す
    gray = gray.astype(np.float32)
    disk = gray > max(4.0, np.percentile(gray, 8))
    background = cv2.GaussianBlur(gray, (0, 0), sigmaX=80, sigmaY=80)
    flattened = gray / np.maximum(background, 1.0)
    flat_u8 = robust_uint8(flattened, disk, 1.0, 99.0)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(16, 16)).apply(flat_u8)
    raw_u8 = robust_uint8(gray, disk, 0.5, 99.5)
    dark_detail = cv2.morphologyEx(clahe, cv2.MORPH_BLACKHAT, np.ones((31, 31), np.uint8))
    dark_detail = robust_uint8(dark_detail, disk, 0.0, 99.5)
    out = np.dstack([raw_u8, clahe, dark_detail])
    out[~disk] = 0
    return out


def prepare_preprocessed(file_name, raw_dir):
    """Drive キャッシュがあればコピー、無ければ raw JPEG から inline で前処理を計算。"""
    local = CFG.PREPROC_DIR / file_name
    if local.exists():
        return
    if CFG.PREPROC_CACHE is not None and (CFG.PREPROC_CACHE / file_name).exists():
        shutil.copy2(CFG.PREPROC_CACHE / file_name, local)
        return
    gray = cv2.imread(str(raw_dir / file_name), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise FileNotFoundError(raw_dir / file_name)
    cv2.imwrite(str(local), solar_three_channel(gray), [cv2.IMWRITE_JPEG_QUALITY, 96])


def load_frozen_folds(path, train_files, n_folds, expected_sha256):
    import csv
    import hashlib
    import io
    import re

    if type(n_folds) is not int or n_folds < 2:
        raise ValueError("n_folds must be an integer >= 2")
    if (not isinstance(train_files, list) or not train_files
            or any(not isinstance(name, str) or not name or name in (".", "..")
                   or name.strip() != name or any(c in name for c in "/\\:")
                   or any(ord(c) < 32 or ord(c) == 127 for c in name) for name in train_files)
            or len(set(train_files)) != len(train_files)):
        raise ValueError("Expected observations must be unique safe basenames")
    if not isinstance(expected_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise ValueError("Provide the externally frozen grouped_folds.csv SHA256")
    payload = Path(path).read_bytes()
    actual_sha256 = hashlib.sha256(payload).hexdigest()
    if actual_sha256 != expected_sha256:
        raise ValueError("Frozen fold CSV SHA256 differs")
    reader = csv.DictReader(io.StringIO(payload.decode("utf-8-sig"), newline=""), strict=True)
    fields = reader.fieldnames
    if (not fields or len(fields) != len(set(fields))
            or not {"file_name", "fold"}.issubset(fields)):
        raise ValueError("Fold CSV needs unique headers including file_name and fold")
    expected, fold_of_file = set(train_files), {}
    try:
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError("Malformed fold CSV row")
            name, value = row["file_name"], row["fold"]
            if name not in expected or name in fold_of_file:
                raise ValueError("Duplicate or unexpected fold observation")
            if not re.fullmatch(r"0|[1-9][0-9]*", value):
                raise ValueError("Fold must be an exact nonnegative integer")
            fold = int(value)
            if fold >= n_folds:
                raise ValueError("Fold is outside configured range")
            fold_of_file[name] = fold
    except csv.Error as error:
        raise ValueError("Malformed fold CSV") from error
    if set(fold_of_file) != expected or set(fold_of_file.values()) != set(range(n_folds)):
        raise ValueError("Fold CSV must cover every observation and every configured fold")
    return fold_of_file, actual_sha256

DATA_ROOT = find_dataset_root()
TRAIN_JSON = next(DATA_ROOT.rglob("MAGFiLO_1.0_Annotations_kaggle2026_train.json"))
with open(TRAIN_JSON, encoding="utf-8") as handle:
    coco = json.load(handle)

image_ids = [item["id"] for item in coco["images"]]
if (any(not (type(image_id) is int or
             (isinstance(image_id, str) and image_id.strip())) for image_id in image_ids)
        or len(set(image_ids)) != len(image_ids)):
    raise ValueError("COCO image IDs must be unique integers or nonempty strings; labeler filenames may repeat")
images = {item["id"]: item for item in coco["images"]}
annotations = defaultdict(list)
for ann in coco["annotations"]:
    annotations[ann["image_id"]].append(ann)
file_to_ids = defaultdict(list)
for image_id, info in images.items():
    file_to_ids[info["file_name"]].append(image_id)
train_files = sorted(file_to_ids)

grouped_folds_path = CFG.ARTIFACT_ROOT / "grouped_folds.csv"
fold_of_file, frozen_fold_sha256 = load_frozen_folds(
    grouped_folds_path, train_files, CFG.N_FOLDS, CFG.GROUPED_FOLDS_SHA256)
print({"frozen_fold_csv": str(grouped_folds_path), "sha256": frozen_fold_sha256})

TRAIN_DIR = find_train_dir(DATA_ROOT)
for name in tqdm(train_files, desc="前処理画像をローカルへ"):
    prepare_preprocessed(name, TRAIN_DIR)

print({"観測": len(train_files),
       "fold別": {f: sum(v == f for v in fold_of_file.values()) for f in (0, 1, 2)},
       "annotation": len(coco["annotations"])})


## (B) YOLO11-seg検出器の3-fold学習

各foldにつき、そのfoldを検証専用に除外し、残り2foldの**全アノテータ
分の独立ラベル**(同一画像でもアノテータごとに別サンプルとして学習)で
学習する。これによりリークなしOOF評価が可能になる。学習済みfoldは
`TRAINING_COMPLETE`マーカーで検出しスキップする。

In [ ]:
# YOLOデータセット構築(all-labeler方式)とtrain関数
def polygon_line(ann, width, height):
    polygons = [p for p in ann.get("segmentation", []) if len(p) >= 6]
    if not polygons:
        return None
    polygon = max(polygons, key=len)  # MAGFiLOは1個体1polygon。複数返る不正データへの安全策
    points = np.asarray(polygon, dtype=np.float32).reshape(-1, 2)
    points[:, 0] = np.clip(points[:, 0] / width, 0, 1)
    points[:, 1] = np.clip(points[:, 1] / height, 0, 1)
    return "0 " + " ".join(f"{value:.6f}" for value in points.reshape(-1))

def write_variant(fold_root, split, file_name, image_id, suffix):
    info = images[image_id]
    stem = Path(file_name).stem + suffix
    image_out = fold_root / "images" / split / f"{stem}.jpeg"
    label_out = fold_root / "labels" / split / f"{stem}.txt"
    image_out.parent.mkdir(parents=True, exist_ok=True)
    if not image_out.exists():
        shutil.copy2((CFG.PREPROC_DIR / file_name).resolve(), image_out)
    lines = [polygon_line(ann, info["width"], info["height"])
             for ann in annotations[image_id] if not ann.get("iscrowd", 0)]
    label_out.parent.mkdir(parents=True, exist_ok=True)
    label_out.write_text("\n".join(line for line in lines if line), encoding="utf-8")

def build_fold_dataset(fold):
    root = CFG.YOLO_ROOT / f"fold_{fold}"
    reference_id = {name: file_to_ids[name][0] for name in train_files}
    for file_name in tqdm(train_files, desc=f"fold {fold} データセット構築"):
        if fold_of_file[file_name] == fold:
            write_variant(root, "val", file_name, reference_id[file_name], "")
        else:
            for image_id in file_to_ids[file_name]:
                write_variant(root, "train", file_name, image_id, f"__labeler_{image_id}")
    yaml_path = root / "data.yaml"
    yaml_path.write_text(f"path: {root}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n",
                          encoding="utf-8")
    return yaml_path

def train_yolo_fold(fold):
    run_dir = CFG.RUNS_ROOT / f"yolo_fold_{fold}"
    best_path = run_dir / "weights" / "best.pt"
    done_path = run_dir / "TRAINING_COMPLETE"
    if done_path.exists() and best_path.exists():
        print(f"fold {fold}: 学習済みcheckpointを再利用 {best_path}")
        return best_path
    pretrained = resolve_pretrained([f"*yolo*fold{fold}*best*.pt", f"*fold{fold}*best*.pt",
                                     f"*yolo_fold_{fold}*best*.pt"])
    if pretrained is not None:
        best_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(pretrained, best_path)
        done_path.write_text("ok", encoding="utf-8")
        print(f"fold {fold}: 添付済み重みを使用 {pretrained}")
        return best_path
    yaml_path = build_fold_dataset(fold)
    model = YOLO(CFG.YOLO_MODEL)
    model.train(
        data=str(yaml_path), epochs=CFG.YOLO_EPOCHS, imgsz=CFG.YOLO_IMGSZ,
        batch=CFG.YOLO_BATCH, device=0, workers=4, patience=CFG.YOLO_PATIENCE,
        seed=CFG.SEED + fold, deterministic=True, single_cls=True,
        mask_ratio=CFG.YOLO_MASK_RATIO, overlap_mask=False, amp=True,
        cache="disk",
        # 査読(2026-08-31)で指摘: これらは "Ultralytics defaults" ではなく
        # build_v30_prize_colab.pyで明示的に設定した値。省略すると別ハイパラで
        # 再学習することになり再現性が壊れるため、必ず全部渡すこと。
        optimizer="AdamW", lr0=7e-4, lrf=0.05, weight_decay=5e-4, warmup_epochs=3,
        degrees=180, translate=0.04, scale=0.25, fliplr=0.5, flipud=0.5,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.18, mosaic=0.20, close_mosaic=10,
        project=str(CFG.RUNS_ROOT), name=f"yolo_fold_{fold}", exist_ok=True,
    )
    done_path.write_text("ok", encoding="utf-8")
    return best_path

yolo_weight_paths = {fold: train_yolo_fold(fold) for fold in (0, 1, 2)}
print({fold: str(path) for fold, path in yolo_weight_paths.items()})


## (C) EdgeAttNet風refiner(EG-MHSA)の学習

境界refinerのボトルネックへEdge-Guided Multi-Head Self-Attentionを
追加した`MediSegEdgeAttUNet`(arXiv:2509.02964の独自再実装、著者コードは
未検証)を、各foldで(検証専用に除外して)学習し、続けてfold0の
checkpointをウォームスタートして全707画像へ低LR・短epochで
fine-tuneする(提出モデル用)。

In [ ]:
# MediSegUNet / EdgeGuidedMHSA / MediSegEdgeAttUNet 定義
"""
MediSeg-AI Knowledge Port for Solar Filament Segmentation
Includes:
1. MediSeg UNet
2. MediSeg DeepLabV3Plus with ASPP (Atrous Spatial Pyramid Pooling)
3. CombinedLoss (BCE + Dice Loss)
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

# ============================================================
# 1. Combined Loss (BCE + Dice Loss)
# ============================================================
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs = probs.view(-1)
        targets = targets.view(-1)
        
        intersection = (probs * targets).sum()
        dice = (2.0 * intersection + self.smooth) / (probs.sum() + targets.sum() + self.smooth)
        return 1.0 - dice

class CombinedLoss(nn.Module):
    def __init__(self, bce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.bce_weight * bce_loss + self.dice_weight * dice_loss

# ============================================================
# 2. MediSeg UNet Architecture
# ============================================================
class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, mid_channels: Optional[int] = None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.double_conv(x)

class Down(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.maxpool_conv(x)

class Up(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, bilinear: bool = True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class MediSegUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1, bilinear: bool = False):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

# ============================================================
# 3. MediSeg DeepLabV3Plus with ASPP Module
# ============================================================
class ASPP(nn.Module):
    def __init__(self, in_channels: int, out_channels: int = 256):
        super().__init__()
        self.conv1x1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn1x1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6, bias=False)
        self.bn3x3_1 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_2 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12, bias=False)
        self.bn3x3_2 = nn.BatchNorm2d(out_channels)
        
        self.conv3x3_3 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18, bias=False)
        self.bn3x3_3 = nn.BatchNorm2d(out_channels)
        
        self.global_avg_pool = nn.AdaptiveAvgPool2d(1)
        self.conv_global = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn_global = nn.BatchNorm2d(out_channels)
        
        self.conv_final = nn.Conv2d(out_channels * 5, out_channels, kernel_size=1, bias=False)
        self.bn_final = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        size = x.shape[2:]
        x1 = self.relu(self.bn1x1(self.conv1x1(x)))
        x2 = self.relu(self.bn3x3_1(self.conv3x3_1(x)))
        x3 = self.relu(self.bn3x3_2(self.conv3x3_2(x)))
        x4 = self.relu(self.bn3x3_3(self.conv3x3_3(x)))
        
        x5 = self.global_avg_pool(x)
        x5 = self.relu(self.bn_global(self.conv_global(x5)))
        x5 = F.interpolate(x5, size=size, mode='bilinear', align_corners=True)
        
        x_cat = torch.cat([x1, x2, x3, x4, x5], dim=1)
        return self.relu(self.bn_final(self.conv_final(x_cat)))

class MediSegDeepLabV3Plus(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 1):
        super().__init__()
        # Feature Extractor
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 512, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True)
        )
        self.aspp = ASPP(512, 256)
        self.classifier = nn.Sequential(
            nn.Conv2d(256, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, out_channels, kernel_size=1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        input_size = x.shape[2:]
        feat = self.stem(x)
        aspp_feat = self.aspp(feat)
        out = self.classifier(aspp_feat)
        return F.interpolate(out, size=input_size, mode='bilinear', align_corners=True)

# ============================================================
# 4. Edge-Guided Attention U-Net (EG-MHSA)
# ============================================================
# 2026-08-29、WINNING_STRATEGY_2026-08-29.md §1.4のPhase 2実験用に追加。
# 参考文献: "EdgeAttNet: Towards Barb-Aware Filament Segmentation"
# (arXiv:2509.02964)。MAGFiLO上でU-Net(mIoU_pairwise 0.5724)から
# EdgeAttNet(0.6451)への改善を報告しており、barb(フィラメントの微細な
# 枝構造)の認識精度向上を狙ったアーキテクチャ。本プロジェクトの
# refiner(MediSegUNet)が抱える主要な弱点(near-miss境界品質)と直接
# 対応するため、置き換え候補として追試する。
#
# 【重要な注意】著者の公開リポジトリ(github.com/dasjar/EdgeAttNet)を
# 直接確認したところ、`model/edgeattnet_model.py`(本来UNetEdgeTransformer
# クラスが定義されているべきファイル)の中身が誤って`main.py`と同一に
# なっており、実際のアーキテクチャコードはリポジトリ上に存在しない
# (2026-08-29、`gh api`でリポジトリ全ファイルを走査して確認)。学習済み
# 重みの配布も確認できなかった。**以下は論文本文のアーキテクチャ記述
# (edge prior E=σ(Conv3x3(入力画像))をbottleneckへbilinear補間し
# 1x1convで射影、Q・Kへ加算、位置エンコーディングなし)のみを根拠にした
# 独自の再実装であり、著者オリジナル実装とのコード上の一致は未検証。**
# ベースにするU-Net本体はMediSegUNetをそのまま流用しているため、
# パラメータ総数は論文記載の値(22.6M)とは一致しない。


class EdgeGuidedMHSA(nn.Module):
    """Edge-Guided Multi-Head Self-Attention (EG-MHSA)。

    U-NetのbottleneckでSelf-Attentionを行う際、入力画像から抽出した
    エッジ事前分布をQuery/Keyに加算し、境界・barb構造への感度を高める。
    Value・出力チャネル数はbottleneckの特徴マップと同一に保つ
    (MediSegUNetへdrop-inできるよう設計)。
    """

    def __init__(self, in_channels: int, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        # E = sigmoid(Conv3x3(入力画像)): 単一チャンネルのエッジ事前分布
        self.edge_conv = nn.Conv2d(in_channels, 1, kernel_size=3, padding=1)
        # E' = Conv1x1(interpolate(E)): bottleneckのチャンネル数へ射影
        self.edge_proj = nn.Conv2d(1, bottleneck_channels, kernel_size=1)
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor, original_image: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        edge_map = torch.sigmoid(self.edge_conv(original_image))
        edge_map = F.interpolate(edge_map, size=(h, w), mode="bilinear", align_corners=True)
        edge_prior = self.edge_proj(edge_map)  # (B, C, H, W)

        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        edge_flat = edge_prior.flatten(2).transpose(1, 2)  # (B, N, C)
        qk = x_flat + edge_flat  # Q = K = x + E'。位置エンコーディングは使わない(論文通り)

        attn_out, _ = self.mhsa(qk, qk, x_flat)  # V = x(エッジ由来の加算なし)
        attn_out = self.norm(attn_out + x_flat)  # 残差接続 + LayerNorm(論文に明記はないが標準的なTransformerブロックとして採用)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegEdgeAttUNet(nn.Module):
    """MediSegUNetのbottleneckにEdgeGuidedMHSAを挿入したdrop-in代替。

    入出力インターフェースはMediSegUNetと完全に同一(in_channels=3の
    crop、out_channels=1の確率マップ)。既存のrefiner学習・推論コードを
    変更せず、モデルクラスの差し替えだけで比較できるように設計している。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.edge_attn = EdgeGuidedMHSA(in_channels, bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.edge_attn(x5, x)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


# ============================================================
# 5. Plain MHSA U-Net (matched-parameter ablation control, no edge prior)
# ============================================================
# 2026-08-31、report/main.tex Future Work item 1(matched-parameter
# ablation)のために追加。EdgeGuidedMHSAからedge_conv/edge_projを除いた
# だけの構成——Q=K=V=bottleneck特徴量の標準的なself-attentionで、
# edge priorをQ/Kへ加算する項だけがない。edge_conv/edge_projは
# 3(in_channels)*1*3*3+1 + 1*1024*1*1+1024 ≈ 2,076パラメータしかなく
# 全体(約35.2M)の0.006%程度なので、このクラスはMediSegEdgeAttUNetと
# ほぼ完全にパラメータ数が一致する("U-Net + MHSA, no positional
# encoding"に相当、EdgeAttNet論文のablationと同じ切り分け)。
# report/main.texが指摘してきた「edge priorそのものの効果か、単に
# attention容量が増えただけか」という交絡を分離するための唯一の差分。


class PlainMHSA(nn.Module):
    """edge priorなしの標準Multi-Head Self-Attention(bottleneck上)。

    EdgeGuidedMHSAと同じchannel数・head数・残差接続+LayerNormを保ちつつ、
    Q・Kへの edge_prior 加算だけを行わない(Q=K=V=bottleneck特徴量)。
    """

    def __init__(self, bottleneck_channels: int, num_heads: int = 8):
        super().__init__()
        if bottleneck_channels % num_heads != 0:
            raise ValueError(
                f"bottleneck_channels({bottleneck_channels}) must be divisible "
                f"by num_heads({num_heads})"
            )
        self.mhsa = nn.MultiheadAttention(
            bottleneck_channels, num_heads, batch_first=True
        )
        self.norm = nn.LayerNorm(bottleneck_channels)

    def forward(self, bottleneck: torch.Tensor) -> torch.Tensor:
        b, c, h, w = bottleneck.shape
        x_flat = bottleneck.flatten(2).transpose(1, 2)  # (B, N, C)
        attn_out, _ = self.mhsa(x_flat, x_flat, x_flat)  # Q=K=V=x(edge priorなし)
        attn_out = self.norm(attn_out + x_flat)
        return attn_out.transpose(1, 2).reshape(b, c, h, w)


class MediSegPlainMHSAUNet(nn.Module):
    """MediSegEdgeAttUNetからedge priorだけを除いた matched-parameter 対照群。

    入出力インターフェースはMediSegUNet/MediSegEdgeAttUNetと完全に同一。
    """

    def __init__(
        self,
        in_channels: int = 3,
        out_channels: int = 1,
        bilinear: bool = False,
        num_heads: int = 8,
    ):
        super().__init__()
        self.inc = DoubleConv(in_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        bottleneck_channels = 1024 // factor
        self.down4 = Down(512, bottleneck_channels)
        self.plain_attn = PlainMHSA(bottleneck_channels, num_heads=num_heads)
        self.up1 = Up(1024, 512 // factor, bilinear)
        self.up2 = Up(512, 256 // factor, bilinear)
        self.up3 = Up(256, 128 // factor, bilinear)
        self.up4 = Up(128, 64, bilinear)
        self.outc = nn.Conv2d(64, out_channels, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x5 = self.plain_attn(x5)
        out = self.up1(x5, x4)
        out = self.up2(out, x3)
        out = self.up3(out, x2)
        out = self.up4(out, x1)
        return self.outc(out)


In [ ]:
# 形状サニティチェック
_probe = MediSegEdgeAttUNet(3, 1, False).cuda()
_dummy = torch.randn(2, 3, CFG.CROP_SIZE, CFG.CROP_SIZE).cuda()
with torch.no_grad():
    _out = _probe(_dummy)
assert _out.shape == (2, 1, CFG.CROP_SIZE, CFG.CROP_SIZE), "出力形状が想定と異なる"
print({"output_shape": tuple(_out.shape),
       "params": sum(p.numel() for p in _probe.parameters())})
del _probe, _dummy, _out
torch.cuda.empty_cache()


In [ ]:
# Dataset・loss・学習ループ定義(fold引数で汎用化)
def square_box_from_bbox(bbox, height, width, context=CFG.CROP_CONTEXT, minimum=96):
    x, y, w, h = map(float, bbox)
    cx, cy = x + w / 2, y + h / 2
    side = int(math.ceil(max(w, h) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

def ann_mask(image_id, ann):
    info = images[image_id]
    height, width = int(info["height"]), int(info["width"])
    segmentation = ann["segmentation"]
    if isinstance(segmentation, list):
        rle = mask_utils.merge(mask_utils.frPyObjects(segmentation, height, width))
    elif isinstance(segmentation.get("counts"), list):
        rle = mask_utils.frPyObjects(segmentation, height, width)
    else:
        rle = segmentation
    return mask_utils.decode(rle).astype(np.uint8)

class InstanceCropDataset(Dataset):
    def __init__(self, file_names, all_labelers=True, augment=False):
        self.samples, self.augment = [], augment
        for file_name in file_names:
            for image_id in (file_to_ids[file_name] if all_labelers else []):
                for ann in annotations[image_id]:
                    if not ann.get("iscrowd", 0) and ann.get("area", 0) > 0:
                        self.samples.append((file_name, image_id, ann))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        file_name, image_id, ann = self.samples[index]
        info = images[image_id]
        image = cv2.imread(str(CFG.PREPROC_DIR / file_name), cv2.IMREAD_COLOR)
        target = ann_mask(image_id, ann)
        x0, y0, x1, y1 = square_box_from_bbox(ann["bbox"], info["height"], info["width"])
        image = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
        target = cv2.resize(target[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_NEAREST)
        image = torch.from_numpy(image.transpose(2, 0, 1).copy()).float() / 255.0
        target = torch.from_numpy(target[None].copy()).float()
        if self.augment:
            k = random.randrange(4)
            image, target = torch.rot90(image, k, (1, 2)), torch.rot90(target, k, (1, 2))
            if random.random() < 0.5:
                image, target = torch.flip(image, (2,)), torch.flip(target, (2,))
            if random.random() < 0.5:
                image, target = torch.flip(image, (1,)), torch.flip(target, (1,))
            if random.random() < 0.5:
                image[:2] = torch.clamp(image[:2] ** random.uniform(0.85, 1.20), 0, 1)
        return image, target

def soft_dice_loss(logits, targets):
    probs = logits.sigmoid()
    intersection = (probs * targets).sum((1, 2, 3))
    denominator = probs.sum((1, 2, 3)) + targets.sum((1, 2, 3))
    return (1 - (2 * intersection + 1) / (denominator + 1)).mean()

def boundary_map(tensor):
    dilation = F.max_pool2d(tensor, 5, stride=1, padding=2)
    erosion = -F.max_pool2d(-tensor, 5, stride=1, padding=2)
    return torch.clamp(dilation - erosion, 0, 1)

def refiner_loss(logits, targets):
    region = 0.35 * F.binary_cross_entropy_with_logits(logits, targets) + 0.65 * soft_dice_loss(logits, targets)
    edge = F.l1_loss(boundary_map(logits.sigmoid()), boundary_map(targets))
    return region + 0.15 * edge

def crop_dice(model, loader, threshold=0.5):
    model.eval()
    total = count = 0
    with torch.no_grad():
        for image, target in loader:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True).bool()
            pred = model(image).sigmoid() >= threshold
            intersection = (pred & target).sum((1, 2, 3)).float()
            denominator = pred.sum((1, 2, 3)) + target.sum((1, 2, 3))
            total += ((2 * intersection + 1) / (denominator + 1)).sum().item()
            count += len(image)
    return total / max(count, 1)

def choose_batch_size(dataset):
    for candidate in CFG.REFINER_BATCH_CANDIDATES:
        try:
            loader = DataLoader(dataset, batch_size=candidate, shuffle=True, num_workers=2, pin_memory=True)
            image, target = next(iter(loader))
            probe = MediSegEdgeAttUNet(3, 1, False).cuda()
            with torch.amp.autocast("cuda"):
                loss = refiner_loss(probe(image.cuda()), target.cuda())
            loss.backward()
            del probe, image, target, loss, loader
            torch.cuda.empty_cache()
            return candidate
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
    raise RuntimeError("どのbatch sizeもGPUメモリに収まらない")

def train_refiner_fold(fold, epochs=CFG.REFINER_EPOCHS):
    checkpoint_path = CFG.ARTIFACT_ROOT / f"mediseg_edgeattnet_refiner_fold{fold}_resume.pt"
    if checkpoint_path.exists():
        state = torch.load(checkpoint_path, map_location="cpu")
        if state.get("epoch", -1) + 1 >= epochs:
            print(f"fold {fold}: 学習済みcheckpointを再利用 {checkpoint_path}")
            return checkpoint_path

    pretrained = resolve_pretrained([f"*refiner*fold{fold}*resume*.pt",
                                     f"*refiner*fold{fold}*.pt"])
    if pretrained is not None and pretrained.resolve() != checkpoint_path.resolve():
        state = torch.load(pretrained, map_location="cpu")
        if state.get("epoch", -1) + 1 >= epochs:
            shutil.copy2(pretrained, checkpoint_path)
            print(f"fold {fold}: 添付済み重みを使用 {pretrained}")
            return checkpoint_path

    train_files_fold = [n for n in train_files if fold_of_file[n] != fold]
    valid_files_fold = [n for n in train_files if fold_of_file[n] == fold]
    train_ds = InstanceCropDataset(train_files_fold, augment=True)
    valid_ds = InstanceCropDataset(valid_files_fold, augment=False)
    batch_size = choose_batch_size(train_ds)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

    model = MediSegEdgeAttUNet(3, 1, False).cuda()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda")
    best_dice, best_state = -1.0, None
    for epoch in range(epochs):
        model.train()
        running = 0.0
        progress = tqdm(train_loader, desc=f"fold{fold} refiner {epoch + 1}/{epochs}")
        for image, target in progress:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = refiner_loss(model(image), target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            progress.set_postfix(loss=f"{running / (progress.n + 1):.4f}")
        scheduler.step()
        dice = crop_dice(model, valid_loader)
        if dice > best_dice:
            best_dice, best_state = dice, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({"epoch": epoch, "model": model.state_dict(), "best_model": best_state,
                    "best_dice": best_dice}, checkpoint_path)
        print(f"fold{fold} epoch {epoch + 1}: crop Dice {dice:.6f} (best {best_dice:.6f})")
    return checkpoint_path

for fold in (0, 1, 2):
    train_refiner_fold(fold)


In [ ]:
# all-data fine-tune(fold0からウォームスタート、低LR・短epoch、提出モデル用)
ALLDATA_CKPT = CFG.ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_all_data_resume.pt"
# all-data fine-tune — 添付重み → Drive ckpt → 再学習 の順で解決
pretrained_refiner = resolve_pretrained(["*refiner*all*data*.pt", "*edgeattnet*all*.pt"])
if pretrained_refiner is not None and not ALLDATA_CKPT.exists():
    ALLDATA_CKPT.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(pretrained_refiner, ALLDATA_CKPT)
    print(f"添付済みrefiner重みを使用 {pretrained_refiner}")
if ALLDATA_CKPT.exists():
    print(f"学習済みcheckpointを再利用 {ALLDATA_CKPT}")
else:
    all_ds = InstanceCropDataset(train_files, augment=True)
    batch_size = choose_batch_size(all_ds)
    loader = DataLoader(all_ds, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)

    warm_start_path = CFG.ARTIFACT_ROOT / "mediseg_edgeattnet_refiner_fold0_resume.pt"
    model = MediSegEdgeAttUNet(3, 1, False).cuda()
    warm_state = torch.load(warm_start_path, map_location="cpu")
    model.load_state_dict(warm_state.get("best_model") or warm_state["model"], strict=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.REFINER_LR * 0.35, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG.REFINER_ALLDATA_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")
    for epoch in range(CFG.REFINER_ALLDATA_EPOCHS):
        model.train()
        running = 0.0
        progress = tqdm(loader, desc=f"all-data refiner {epoch + 1}/{CFG.REFINER_ALLDATA_EPOCHS}")
        for image, target in progress:
            image, target = image.cuda(non_blocking=True), target.cuda(non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = refiner_loss(model(image), target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running += loss.item()
            progress.set_postfix(loss=f"{running / (progress.n + 1):.4f}")
        scheduler.step()
        print(f"epoch {epoch + 1}: mean loss {running / len(loader):.4f}")
    torch.save({"model": model.state_dict(), "best_model": model.state_dict(),
                "warm_start_from": str(warm_start_path)}, ALLDATA_CKPT)
    print(f"saved: {ALLDATA_CKPT}")


## (D) test推論・提出物生成

提出構成(V52): fold2単体YOLO(アンサンブルなし、V50/V52で3-foldアンサンブルより
同等以上と確認済み) + all-data EdgeAttNet refiner + shrink@0.45 +
drop_contained_then_area_asc。

In [ ]:
# test画像一覧、単一モデル推論、refiner適用、排他化、submission CSV生成
def find_test_dir():
    return next(DATA_ROOT.rglob("test_images"))

TEST_DIR = find_test_dir()
test_files = sorted(p.name for p in TEST_DIR.glob("*.jpeg"))
assert len(test_files) == 180, f"test画像が180枚ではありません: {len(test_files)}"
for file_name in tqdm(test_files, desc="test画像をローカルへ"):
    prepare_preprocessed(file_name, TEST_DIR)

yolo_model = YOLO(str(yolo_weight_paths[CFG.SINGLE_FOLD]))
if CFG.SUBMISSION_VARIANT == "plain_mhsa_f2":
    plain_ckpt = resolve_pretrained(
        ["*plain_mhsa*fold2*.pt", "*plain_mhsa*.pt"])
    if plain_ckpt is None:
        raise FileNotFoundError(
            "plain_mhsa_f2 の refiner 重みが見つかりません。"
            "重み dataset をアタッチするか Drive artifacts に配置してください")
    refiner = MediSegPlainMHSAUNet(3, 1, False).cuda().eval()
    refiner_ckpt = torch.load(plain_ckpt, map_location="cuda", weights_only=False)
else:
    refiner = MediSegEdgeAttUNet(3, 1, False).cuda().eval()
    refiner_ckpt = torch.load(ALLDATA_CKPT, map_location="cuda", weights_only=False)
refiner.load_state_dict(refiner_ckpt.get("best_model") or refiner_ckpt["model"], strict=True)

def result_masks(result):
    if result.masks is None:
        return []
    height, width = result.orig_shape
    output = []
    for mask in result.masks.data.detach().cpu().numpy():
        if mask.shape != (height, width):
            mask = cv2.resize(mask, (width, height), interpolation=cv2.INTER_LINEAR)
        output.append(mask >= 0.5)
    return output

def single_fold_proposals(image_path):
    result = yolo_model.predict(str(image_path), imgsz=CFG.YOLO_IMGSZ,
                                 conf=max(0.08, CFG.SUBMISSION_CONF * 0.75), iou=CFG.PREDICT_IOU,
                                 agnostic_nms=True, retina_masks=True, verbose=False)[0]
    masks = result_masks(result)
    confidences = result.boxes.conf.detach().cpu().numpy().tolist() if result.boxes is not None else []
    return [(s, m) for s, m in zip(confidences, masks) if s >= CFG.SUBMISSION_CONF]

@torch.no_grad()
def refine_proposal(image, proposal):
    x0, y0, x1, y1 = mask_square_box(proposal)
    crop = cv2.resize(image[y0:y1, x0:x1], (CFG.CROP_SIZE, CFG.CROP_SIZE), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(crop.transpose(2, 0, 1)[None].copy()).float().cuda() / 255.0
    probability = refiner(tensor).sigmoid()[0, 0].cpu().numpy()
    probability = cv2.resize(probability, (x1 - x0, y1 - y0), interpolation=cv2.INTER_LINEAR)
    refined = np.zeros_like(proposal, dtype=bool)
    refined[y0:y1, x0:x1] = probability >= CFG.REFINER_THRESHOLD
    return refined & proposal

def mask_square_box(mask, context=1.65, minimum=96):
    height, width = mask.shape
    ys, xs = np.where(mask)
    if not len(xs):
        return 0, 0, width, height
    cx, cy = (xs.min() + xs.max() + 1) / 2, (ys.min() + ys.max() + 1) / 2
    side = int(math.ceil(max(xs.max() - xs.min() + 1, ys.max() - ys.min() + 1) * context))
    side = min(max(side, minimum), min(height, width))
    x0, y0 = int(round(cx - side / 2)), int(round(cy - side / 2))
    x1, y1 = x0 + side, y0 + side
    if x0 < 0: x1 -= x0; x0 = 0
    if y0 < 0: y1 -= y0; y0 = 0
    if x1 > width: x0 -= x1 - width; x1 = width
    if y1 > height: y0 -= y1 - height; y1 = height
    return max(0, x0), max(0, y0), min(width, x1), min(height, y1)

def greedy_disjoint(masks, order, min_area=CFG.MIN_AREA):
    occupied = np.zeros((2048, 2048), dtype=bool)
    kept = []
    for index in order:
        exclusive = masks[index] & ~occupied
        if exclusive.sum() < min_area:
            continue
        kept.append(exclusive)
        occupied |= exclusive
    return kept

def drop_contained_indices(masks, threshold=CFG.DROP_THRESHOLD):
    if len(masks) < 2:
        return list(range(len(masks)))
    rles = mask_utils.encode(np.asfortranarray(np.stack(masks, axis=-1).astype(np.uint8)))
    areas = np.asarray(mask_utils.area(rles), dtype=np.float64)
    self_iou = np.asarray(mask_utils.iou(rles, rles, [0] * len(rles)), dtype=np.float64)
    dropped = set()
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            if i in dropped or j in dropped:
                continue
            value = self_iou[i, j]
            if value <= 0:
                continue
            intersection = value * (areas[i] + areas[j]) / (1.0 + value)
            smaller = i if areas[i] <= areas[j] else j
            if intersection / max(areas[smaller], 1.0) >= threshold:
                dropped.add(smaller)
    return [i for i in range(len(masks)) if i not in dropped]

def apply_ownership(masks, scores):
    if not masks:
        return []
    areas = np.array([m.sum() for m in masks], dtype=np.float64)
    subset = drop_contained_indices(masks)
    return greedy_disjoint(masks, sorted(subset, key=lambda i: areas[i]))

rows, audit = [], []
inference_start = time.time()
for file_name in tqdm(test_files, desc="test推論"):
    image_path = CFG.PREPROC_DIR / file_name
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    proposals = single_fold_proposals(image_path)
    scores = np.array([c for c, _ in proposals], dtype=np.float32)
    refined = [refine_proposal(image, m) for _, m in proposals]
    final_masks = apply_ownership(refined, scores)
    image_id = Path(file_name).stem
    for index, mask in enumerate(final_masks, start=1):
        encoded = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
        rows.append({"filament_id": f"{image_id}_{index}",
                     "segmentation_rle": encoded["counts"].decode("utf-8")})
    audit.append({"file_name": file_name, "proposals": len(proposals), "submitted": len(final_masks)})

submission = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
OUTPUT_PATH = (WORK_ROOT / "submission.csv"
               if CFG.SUBMISSION_VARIANT == "prod"
               else WORK_ROOT / f"submission_{CFG.SUBMISSION_VARIANT}.csv")
submission.to_csv(OUTPUT_PATH, index=False, lineterminator="\n")
shutil.copy2(OUTPUT_PATH, CFG.ARTIFACT_ROOT /
               (f"submission_master_pipeline_{CFG.SUBMISSION_VARIANT}.csv"
                if CFG.SUBMISSION_VARIANT != "prod"
                else "submission_master_pipeline.csv"))
total_s = time.time() - inference_start
efficiency = {"test_images": len(test_files),
              "total_wall_s": round(total_s, 1),
              "s_per_image": round(total_s / len(test_files), 3),
              "gpu": torch.cuda.get_device_name(0)}
(CFG.ARTIFACT_ROOT / "submission_efficiency.json").write_text(
    json.dumps(efficiency, indent=2), encoding="utf-8")
print(efficiency)
print({"rows": len(submission), "images_with_predictions":
       len(set(a["file_name"] for a in audit if a["submitted"] > 0))})


## (E) 公式RLE厳密検証

In [ ]:
# 提出前チェックリスト: 列・重複・空値・重畳ピクセルゼロを確認
checked = pd.read_csv(OUTPUT_PATH, keep_default_na=False)
assert checked.columns.tolist() == ["filament_id", "segmentation_rle"]
assert not checked.isna().any().any()
assert not checked["filament_id"].duplicated().any()
assert not checked["segmentation_rle"].str.strip().eq("").any()

test_ids = {Path(name).stem for name in test_files}
submitted_ids = {value.rsplit("_", 1)[0] for value in checked["filament_id"]}
assert submitted_ids <= test_ids

occupancy = {}
for row in tqdm(checked.itertuples(index=False), total=len(checked), desc="decode + overlap検証"):
    decoded = mask_utils.decode({"size": [2048, 2048], "counts": row.segmentation_rle.encode("utf-8")})
    assert decoded.shape == (2048, 2048) and decoded.sum() > 0
    image_id = row.filament_id.rsplit("_", 1)[0]
    occ = occupancy.setdefault(image_id, np.zeros((2048, 2048), dtype=bool))
    assert not np.logical_and(occ, decoded.astype(bool)).any(), f"重複: {image_id}"
    occ |= decoded.astype(bool)

print({"rows": len(checked), "images": len(submitted_ids), "overlap_pixels": 0,
       "status": "全検証パス。submission.csv は提出可能な状態"})


## (F) 環境固定記録

In [ ]:
# [6/6] 環境固定記録 — pip freeze + 重みSHA256
import hashlib, subprocess, sys

freeze_path = CFG.ARTIFACT_ROOT / "submission_reproduction_pip_freeze.txt"
freeze_path.write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"],
                    capture_output=True, text=True).stdout,
    encoding="utf-8")
print(f"pip freeze -> {freeze_path}")

refiner_label, refiner_path = (
    ("refiner_all_data", ALLDATA_CKPT) if CFG.SUBMISSION_VARIANT == "prod"
    else ("refiner_" + CFG.SUBMISSION_VARIANT, plain_ckpt))
for label, path in [
    ("yolo_fold2", yolo_weight_paths.get(CFG.SINGLE_FOLD)),
    (refiner_label, refiner_path),
]:
    if path and Path(path).exists():
        digest = hashlib.sha256(Path(path).read_bytes()).hexdigest()
        print(f"SHA256 {label}: {digest}")
        (CFG.ARTIFACT_ROOT / f"sha256_{label}.txt").write_text(digest + "\n", encoding="utf-8")
